# Model Training

In this notebook, we train multiple machine learning models using the selected features.

The objectives of this notebook are:

- Split the dataset into training and testing sets.
- Build a preprocessing pipeline.
- Train multiple machine learning models.
- Save the trained models for later evaluation.

> **Note:** Model comparison and interpretability will be performed in separate notebooks.

## Cell 2 — Import Libraries ##

In [22]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier




from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


## Cell 3 — Load Selected Dataset ##

In [23]:
# Load selected dataset
df = pd.read_csv("../data/processed/selected_data.csv")

print("=" * 60)
print("Dataset Loaded Successfully")
print("=" * 60)

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Loaded Successfully
Dataset Shape: (101766, 29)


,num_lab_procedures,diag_1,diag_2,diag_3,num_medications,time_in_hospital,age,discharge_disposition_id,number_diagnoses,num_procedures,...,metformin,glipizide,glyburide,number_emergency,changed_medications,change,pioglitazone,rosiglitazone,glimepiride,readmitted
0,41,250.83,NaN,NaN,1,1,5,25,1,0,...,No,No,No,0,0,No,No,No,No,0
1,59,276,250.01,255,18,3,15,1,9,0,...,No,No,No,0,1,Ch,No,No,No,1
2,11,648,250,V27,13,2,25,1,6,5,...,No,Steady,No,0,0,No,No,No,No,0
3,44,8,250.43,403,16,2,35,1,7,1,...,No,No,No,0,1,Ch,No,No,No,0
4,51,197,157,250,8,1,45,1,5,0,...,No,Steady,No,0,0,Ch,No,No,No,0


## Cell 4 — Separate Features & Target ##

In [24]:
# Separate features and target

X = df.drop(columns=["readmitted"])
y = df["readmitted"]

print("=" * 60)
print("Features & Target")
print("=" * 60)

print(f"Features Shape : {X.shape}")
print(f"Target Shape   : {y.shape}")

Features & Target
Features Shape : (101766, 28)
Target Shape   : (101766,)


## Cell 5 — Train/Test Split ##

In [25]:
# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 60)
print("Train/Test Split")
print("=" * 60)

print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")

Train/Test Split
Training Samples : 81412
Testing Samples  : 20354


## Cell 6 — Identify Feature Types ##

In [26]:
# Identify feature types

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("=" * 60)
print("Feature Summary")
print("=" * 60)

print(f"Numerical Features   : {len(numerical_features)}")
print(f"Categorical Features : {len(categorical_features)}")

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Feature Summary
Numerical Features   : 15
Categorical Features : 13

Numerical Features:
['num_lab_procedures', 'num_medications', 'time_in_hospital', 'age', 'discharge_disposition_id', 'number_diagnoses', 'num_procedures', 'total_visits', 'admission_type_id', 'number_inpatient', 'admission_source_id', 'active_medications', 'number_outpatient', 'number_emergency', 'changed_medications']

Categorical Features:
['diag_1', 'diag_2', 'diag_3', 'race', 'insulin', 'gender', 'metformin', 'glipizide', 'glyburide', 'change', 'pioglitazone', 'rosiglitazone', 'glimepiride']


## Cell 7 — Build Preprocessing Pipeline ##

In [27]:
# Preprocessing for numerical features
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# Preprocessing for categorical features
categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## Cell 8 — Define Machine Learning Models ##

In [28]:
models = {
    "Logistic Regression": LogisticRegression(
        random_state=42,
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        verbose=-1
    ),

    "CatBoost": CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}

## Cell 9 — Import Required Libraries ##

In [29]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [30]:
import time
import joblib
from pathlib import Path

## Cell 10 — Create Models Directory ##

In [31]:
# Create models directory

MODELS_DIR = Path("../models")

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Models directory: {MODELS_DIR.resolve()}")

Models directory: E:\all projects\Data Science\Early Prediction of Hospital Readmission and Patient Risk\models


## Cell 11 — Training Function ##

In [32]:
def train_model(model_name, model, preprocessor,
                X_train, y_train):
    """
    Train a machine learning model using a preprocessing pipeline.

    Parameters
    ----------
    model_name : str
        Name of the model.

    model : estimator
        Scikit-learn estimator.

    preprocessor : ColumnTransformer
        Data preprocessing pipeline.

    X_train : DataFrame
    y_train : Series

    Returns
    -------
    trained_pipeline
    training_time
    """

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    start_time = time.time()

    pipeline.fit(
        X_train,
        y_train
    )

    training_time = time.time() - start_time

    return pipeline, training_time

## Cell 12 — Train All Models ##

In [33]:
trained_models = {}

training_summary = []

for model_name, model in models.items():

    print("=" * 60)
    print(f"Training {model_name}")
    print("=" * 60)

    pipeline, training_time = train_model(
        model_name=model_name,
        model=model,
        preprocessor=preprocessor,
        X_train=X_train,
        y_train=y_train
    )

    trained_models[model_name] = pipeline

    training_summary.append({
        "Model": model_name,
        "Training Time (sec)": round(training_time, 3)
    })

    print(f"Completed in {training_time:.3f} seconds\n")

Training Logistic Regression


Completed in 16.598 seconds

Training Decision Tree
Completed in 35.867 seconds

Training Random Forest
Completed in 189.674 seconds

Training XGBoost
Completed in 3.886 seconds

Training LightGBM
Completed in 2.877 seconds

Training CatBoost
Completed in 43.913 seconds



## Cell 13 — Training Summary ##

In [34]:
training_summary = pd.DataFrame(training_summary)

training_summary

,Model,Training Time (sec)
0,Logistic Regression,16.598
1,Decision Tree,35.867
2,Random Forest,189.674
3,XGBoost,3.886
4,LightGBM,2.877
5,CatBoost,43.913


## Cell 14 — Save Models ##

In [35]:
for model_name, pipeline in trained_models.items():

    filename = (
        model_name
        .lower()
        .replace(" ", "_")
        + ".pkl"
    )

    joblib.dump(
        pipeline,
        MODELS_DIR / filename
    )

print("All trained models have been saved successfully.")

All trained models have been saved successfully.
